In [ ]:
# Load the all-speed charger data (tab 1a)
# 1. Read the CSV from data/raw
# 2. Skip the 2 junk rows so row 3 becomes the headers
# 3. Treat commas in numbers as thousands separators (15,116 → 15116)
# 4. Treat [x] as missing (NaN), not text

# Check the load

In [2]:
import pandas as pd

In [3]:
chargers = pd.read_csv("C:/Users/kraus/Desktop/Ironhack/ev-charging-gap/data/raw/evci9001_1a.csv", skiprows=2, thousands=",", na_values="[x]")

In [4]:
print(chargers.shape)
chargers.head()

(433, 30)


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,...,Apr-24,Jul-24,Oct-24 [Note 14],Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26,Jul-26
0,K02000001,United Kingdom,15116.0,16505.0,17947.0,18265.0,19487.0,20775.0,22790.0,24374.0,...,59670.0,64632.0,70042.0,73334.0,76507.0,82002.0,86021.0,87796.0,92141.0,94595.0
1,K03000001,Great Britain,14821.0,16210.0,17642.0,17953.0,19169.0,20455.0,22463.0,24044.0,...,59125.0,64014.0,69403.0,72654.0,75835.0,81318.0,85283.0,87069.0,91414.0,93864.0
2,E92000001,England,12549.0,13719.0,14979.0,15395.0,16456.0,17459.0,19261.0,20563.0,...,51506.0,55631.0,60364.0,63389.0,65877.0,70672.0,74115.0,75818.0,79198.0,81231.0
3,E12000001,North East,738.0,752.0,786.0,812.0,849.0,820.0,854.0,887.0,...,1942.0,1919.0,2295.0,2583.0,2553.0,2703.0,2698.0,2734.0,2950.0,3090.0
4,E06000047,County Durham,92.0,96.0,102.0,105.0,106.0,110.0,121.0,116.0,...,309.0,313.0,359.0,400.0,462.0,482.0,482.0,489.0,523.0,539.0


In [7]:
# Keep only the lowest geography level (individual councils)
# 1. List the code prefixes for councils in England, Scotland, Wales and NI
# 2. Take the first 3 characters of each area code
# 3. Keep rows whose prefix is in the list
# 4. Check the shape

In [6]:
lowest_tier = ["E06", "E07", "E08", "E09", "S12", "W06", "N09"]

chargers_la = chargers[chargers["Local authority / region code"].str[:3].isin(lowest_tier)]

print(chargers_la.shape)

(385, 30)


In [12]:
print(chargers_la["Oct-19"].sum())
print(chargers_la["Jul-26"].sum())

15116.0
94595.0


In [8]:
# Sanity check: do the councils add up to the UK total?
# 1. Add up chargers across all councils for the first quarter (Oct-19)
# 2. Do the same for the latest quarter (Jul-26)
# 3. Compare both with the published UK row (15116 and 94595)
# 4. If they match, the filter has no gaps and no double counting

In [10]:
# Rename and reshape from wide to long
# 1. Shorten the two label column names to area_code and area_name
# 2. Melt: keep area_code and area_name on every row,
#    turn the 28 quarter columns into rows (one row per council per quarter)
# 3. Check the shape

In [14]:
chargers_la = chargers_la.rename(columns={
    "Local authority / region code": "area_code",
    "Local authority / region name": "area_name"
})

chargers_long = chargers_la.melt(
    id_vars=["area_code", "area_name"],
    var_name="quarter",
    value_name="chargers"
)

print(chargers_long.shape)
chargers_long.head()

(10780, 4)


,area_code,area_name,quarter,chargers
0,E06000047,County Durham,Oct-19,92.0
1,E06000005,Darlington,Oct-19,27.0
2,E06000001,Hartlepool,Oct-19,5.0
3,E06000002,Middlesbrough,Oct-19,25.0
4,E06000057,Northumberland,Oct-19,139.0


In [11]:
# Turn quarter labels into real dates
# 1. Remove the " [Note 14]" footnote marker from the Oct-24 label
# 2. Convert text like "Oct-19" into a date (1 October 2019)

In [15]:
chargers_long["quarter"] = chargers_long["quarter"].str.replace(" [Note 14]", "", regex=False)

chargers_long["date"] = pd.to_datetime(chargers_long["quarter"], format="%b-%y")

chargers_long.head()

,area_code,area_name,quarter,chargers,date
0,E06000047,County Durham,Oct-19,92.0,2019-10-01
1,E06000005,Darlington,Oct-19,27.0,2019-10-01
2,E06000001,Hartlepool,Oct-19,5.0,2019-10-01
3,E06000002,Middlesbrough,Oct-19,25.0,2019-10-01
4,E06000057,Northumberland,Oct-19,139.0,2019-10-01


In [12]:
# Check every label converted

In [16]:
chargers_long["date"].nunique()

28

In [ ]:
# Chargers are counted on the 1st of the quarter and vehicles at the end, 
# so I shifted charger dates back one day to match:

# Line up charger dates with vehicle quarters
# Chargers are counted on the 1st of the quarter, vehicles at the end of the previous one
# 1. Step each date back one day (1 Jul → 30 Jun)
# 2. Find which quarter that day falls in (30 Jun → Q2)
# 3. Write it in the vehicle file's style ("2026 Q2") as join_quarter

In [17]:
chargers_long["join_quarter"] = (chargers_long["date"] - pd.Timedelta(days=1)).dt.to_period("Q").dt.strftime("%Y Q%q")

chargers_long.head()

,area_code,area_name,quarter,chargers,date,join_quarter
0,E06000047,County Durham,Oct-19,92.0,2019-10-01,2019 Q3
1,E06000005,Darlington,Oct-19,27.0,2019-10-01,2019 Q3
2,E06000001,Hartlepool,Oct-19,5.0,2019-10-01,2019 Q3
3,E06000002,Middlesbrough,Oct-19,25.0,2019-10-01,2019 Q3
4,E06000057,Northumberland,Oct-19,139.0,2019-10-01,2019 Q3


In [13]:
# Save the clean table
# 1. Keep 5 columns: area_code, area_name, join_quarter, date, chargers
# 2. Save to data/clean as chargers_all_speeds.csv, without row numbers

In [18]:
chargers_clean = chargers_long[["area_code", "area_name", "join_quarter", "date", "chargers"]]

chargers_clean.to_csv("../data/clean/chargers_all_speeds.csv", index=False)

In [14]:
# Load the rapid charger data (tab 1b)
# Same settings as tab 1a, new name (rapid) so the 1a table isn't overwritten

In [21]:
rapid = pd.read_csv("C:/Users/kraus/Desktop/Ironhack/ev-charging-gap/data/raw/evci9001_1b.csv", skiprows=2, thousands=",", na_values="[x]")

In [ ]:
# Keep only councils, then check against the UK total
# 1. Reuse the lowest_tier list to keep council rows only
# 2. Check the shape
# 3. Add up the first and last quarter columns, using positions not names
#    (tab 1b headers are messy)
# 4. Compare with the UK row

In [22]:
rapid_la = rapid[rapid["Local authority / region code"].str[:3].isin(lowest_tier)]

print(rapid_la.shape)
print(rapid_la.iloc[:, 2].sum())
print(rapid_la.iloc[:, -1].sum())

(385, 30)
2495.0
20255.0


In [16]:
# Rename and reshape from wide to long
# Same as tab 1a, but the number column is called rapid_chargers
# Check the shape

In [23]:
rapid_la = rapid_la.rename(columns={
    "Local authority / region code": "area_code",
    "Local authority / region name": "area_name"
})

rapid_long = rapid_la.melt(
    id_vars=["area_code", "area_name"],
    var_name="quarter",
    value_name="rapid_chargers"
)

print(rapid_long.shape)
rapid_long.head()

(10780, 4)


,area_code,area_name,quarter,rapid_chargers
0,E06000047,County Durham,Oct-19\n(25kW and above),13.0
1,E06000005,Darlington,Oct-19\n(25kW and above),2.0
2,E06000001,Hartlepool,Oct-19\n(25kW and above),0.0
3,E06000002,Middlesbrough,Oct-19\n(25kW and above),4.0
4,E06000057,Northumberland,Oct-19\n(25kW and above),24.0


In [17]:
# Clean the messy quarter labels and turn them into dates
# 1. Split each label at spaces and line breaks, keep the first piece
#    ("Oct-19\n(25kW and above)" → "Oct-19")
# 2. Fix labels where July is spelled out ("July-21" → "Jul-21")
# 3. Convert to real dates
# 4. Check

In [24]:
rapid_long["quarter"] = rapid_long["quarter"].str.split().str[0].str.replace("July", "Jul", regex=False)

rapid_long["date"] = pd.to_datetime(rapid_long["quarter"], format="%b-%y")

print(rapid_long["date"].nunique())
rapid_long.head()

28


,area_code,area_name,quarter,rapid_chargers,date
0,E06000047,County Durham,Oct-19,13.0,2019-10-01
1,E06000005,Darlington,Oct-19,2.0,2019-10-01
2,E06000001,Hartlepool,Oct-19,0.0,2019-10-01
3,E06000002,Middlesbrough,Oct-19,4.0,2019-10-01
4,E06000057,Northumberland,Oct-19,24.0,2019-10-01


In [18]:
# Keep October 2023 onwards and line up with vehicle quarters
# 1. Keep rows dated on or after 1 Oct 2023, when "rapid" changed from 25kW+ to 50kW+
#    (so every row uses the same definition)
# 2. Create join_quarter, the same way as for tab 1a
# 3. Check

In [25]:
rapid_long = rapid_long[rapid_long["date"] >= "2023-10-01"]

rapid_long["join_quarter"] = (rapid_long["date"] - pd.Timedelta(days=1)).dt.to_period("Q").dt.strftime("%Y Q%q")

print(rapid_long.shape)
print(rapid_long["date"].nunique())
rapid_long.head()

(4620, 6)
12


,area_code,area_name,quarter,rapid_chargers,date,join_quarter
6160,E06000047,County Durham,Oct-23,67.0,2023-10-01,2023 Q3
6161,E06000005,Darlington,Oct-23,6.0,2023-10-01,2023 Q3
6162,E06000001,Hartlepool,Oct-23,5.0,2023-10-01,2023 Q3
6163,E06000002,Middlesbrough,Oct-23,16.0,2023-10-01,2023 Q3
6164,E06000057,Northumberland,Oct-23,65.0,2023-10-01,2023 Q3


In [19]:
# Save the clean table
# Same 5-column layout as the all-speed file, saved as rapid_chargers.csv

In [26]:
rapid_clean = rapid_long[["area_code", "area_name", "join_quarter", "date", "rapid_chargers"]]

rapid_clean.to_csv("../data/clean/rapid_chargers.csv", index=False)